In [2]:
import numpy as np
import pandas as pd

np.random.seed(10)
n_rows = 1200

branchen = ["Bäckerei", "Metallverarbeitung", "Supermarkt", "Hotel", "Logistik"]
monate = ["Januar", "Februar", "März"]

data = {
    "Kunden_ID": np.random.randint(100000, 999999, n_rows),
    "Branche": np.random.choice(branchen, n_rows),
    "Monat": np.random.choice(monate, n_rows),
    "Verbrauch_kWh": np.random.randint(4000, 85000, n_rows),
    "Tarif_Cent_kWh": np.random.uniform(26.0, 45.0, n_rows).round(2),
    "Netz_und_Einkauf_Cent_kWh": np.random.uniform(22.0, 38.0, n_rows).round(2),
    "Kundenzufriedenheit_Score": np.random.randint(1, 11, n_rows),  # 1 = unzufrieden, 10 = sehr treu
}

df_raw = pd.DataFrame(data)


df_raw.loc[df_raw.sample(40).index, "Verbrauch_kWh"] = np.nan

df_raw["Monat"] = df_raw["Monat"].replace("Januar", "Januar_FEHLER")

df_raw.to_csv("Rohdaten_Gewerbekunden.csv", sep=";", index=False)


In [3]:
# 1. Datei einlesen
df = pd.read_csv("Rohdaten_Gewerbekunden.csv", sep=";")

# 2. Welche Monate stehen aktuell wirklich in der Spalte?
print("Vorhandene Monate im Datensatz:")
print(df["Monat"].unique())

# 3. Wie viele Werte fehlen in jeder Spalte?
print("\nFehlende Werte pro Spalte:")
print(df.isna().sum())

Vorhandene Monate im Datensatz:
<StringArray>
['Februar', 'Januar_FEHLER', 'März']
Length: 3, dtype: str

Fehlende Werte pro Spalte:
Kunden_ID                     0
Branche                       0
Monat                         0
Verbrauch_kWh                40
Tarif_Cent_kWh                0
Netz_und_Einkauf_Cent_kWh     0
Kundenzufriedenheit_Score     0
dtype: int64


In [ ]:
#Textfehler Januar_FEHLER korrigieren
df["Monat"] = df["Monat"].replace("Januar_FEHLER", "Januar")

#Fehlende Werte in Verbrauch_kWh mit dem Median füllen um Daten zum arbeiten zu haben
durchschnittsverbrauch = df["Verbrauch_kWh"].mean()

df["Verbrauch_kWh"] = df["Verbrauch_kWh"].fillna(durchschnittsverbrauch)

#Überprüfen, ob Korrekturen vorgenommen wurden
print("Bereinigte Monate im Datensatz:")
print(df["Monat"].unique())

print("\nVerbleibende Lücken pro Spalte:")
print(df.isna().sum())

Bereinigte Monate im Datensatz:
<StringArray>
['Februar', 'Januar', 'März']
Length: 3, dtype: str

Verbleibende Lücken pro Spalte:
Kunden_ID                    0
Branche                      0
Monat                        0
Verbrauch_kWh                0
Tarif_Cent_kWh               0
Netz_und_Einkauf_Cent_kWh    0
Kundenzufriedenheit_Score    0
dtype: int64


In [8]:
#Marge pro kWh in Cent berechnen
df["Marge_Cent_kWh"] = df["Tarif_Cent_kWh"] - df["Netz_und_Einkauf_Cent_kWh"].round(2)

#Gesamte Monatsmarge in Euro berechnen
df["Marge_Euro"] = (df["Marge_Cent_kWh"] * df["Verbrauch_kWh"] / 100).round(2)

#Sortieren nach größten Verlusten
df_verlust = df.sort_values(by="Marge_Euro", ascending=True)

df_verlust.head(5)

,Kunden_ID,Branche,Monat,Verbrauch_kWh,Tarif_Cent_kWh,Netz_und_Einkauf_Cent_kWh,Kundenzufriedenheit_Score,Marge_Cent_kWh,Marge_Euro
248,138444,Metallverarbeitung,Februar,80614.0,27.03,36.24,4,-9.21,-7424.55
1098,510703,Logistik,Januar,75088.0,26.98,36.61,4,-9.63,-7230.97
980,256626,Bäckerei,März,83533.0,29.60,37.92,5,-8.32,-6949.95
1020,279431,Bäckerei,März,66893.0,27.44,37.13,5,-9.69,-6481.93
152,460982,Logistik,Februar,56455.0,26.12,37.11,10,-10.99,-6204.40


In [17]:
#Filterung aller Kunden mit Verlusten
df_risiko_all = df[df["Marge_Euro"] < 0]

#Sortierung der größten Verlusten
df_risiko_all = df_risiko_all.sort_values(by="Marge_Euro", ascending=True)

#Minusbeträge rot färben
def farbe_rot_negativ(wert):
    if wert < 0:
        color = "red"
    else:
        color = "black"
    return f"color: {color}"

#Excel auf PC speichern
excel_name = "Risiko_Kunden_Portfolio_Bericht.xlsx"
df_risiko_all.style.map(farbe_rot_negativ, subset=["Marge_Euro"]).to_excel(excel_name, index=False)